# Centralised LSTM fh=1 modified

This notebook reports `avg_normalised_loss`, `avg_actual_loss`, saves base/reconciled/naive forecasts, and evaluates MAE, MASE, RMSSE.

For CLSTM, there is one centralised model, so early stopping is checked once for the global training loop.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from clstm import (
    Datacollector,
    run_centralised,
    compute_metrics_from_dicts,
    make_reconciliation_frames,
    evaluate_reconciliation_results,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
# -------------------------
# Parameter setup
# -------------------------
#args_path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/GEF2017_univarivate.csv'
args_path = "E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv" #'E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv'
args_fh  = 5
args_lags = 48
args_INPUT, args_HIDDEN, args_LAYERS, args_DROP = 1, 64, 2, 0.1
args_EPOCHS, args_BATCH_SIZE, args_LR = 100 + args_fh*50, 256, 1e-3
args_TRAIN_RATIO = 0.8
args_MODEL_NAME = 'clstm'
args_truncated = None
args_SCALING_MODE = 'per_partition'  # 'global' or 'per_partition'
args_DROP_BOUNDARY_GAP = True

# Early stopping settings
args_EARLY_STOP_ENABLED = True
args_EARLY_STOP_TOL = 1e-5
args_MIN_EPOCHS = 20 + args_fh*50

dict_ = Datacollector(pd.read_csv(args_path), lags=args_lags, ts=range(15), fh=args_fh)
df = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
df['ds'] = pd.to_datetime(df['ds'], format='mixed')
if args_truncated is not None:
    df = df[df['ds'] >= args_truncated].reset_index(drop=True)
df = df.reset_index(drop=True)

parts = sorted(df['partition_id'].unique())
lag_cols_reversed = list(reversed(sorted([c for c in df.columns if c.startswith('lags_')], key=lambda x: int(x.split('_')[1]))))
forecast_horizon = ['y'] + sorted([c for c in df.columns if c.startswith('post_')], key=lambda x: int(x.split('_')[1]))
df2 = df[['unique_id', 'partition_id', 'ds'] + lag_cols_reversed + forecast_horizon].copy()

print(df2.head())
print(f'num partitions: {len(parts)}')
print(f'Scaling mode: {args_SCALING_MODE} | drop_boundary_gap: {args_DROP_BOUNDARY_GAP}')
print(f'X shape: {df2[lag_cols_reversed].shape} | y shape: {df2[forecast_horizon].shape}')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 18.29it/s]


  unique_id  partition_id                  ds  lags_48  lags_47  lags_46  \
0     Total             0 2003-03-03 00:00:00  12864.0  12389.0  12155.0   
1     Total             0 2003-03-03 01:00:00  12389.0  12155.0  12072.0   
2     Total             0 2003-03-03 02:00:00  12155.0  12072.0  12162.0   
3     Total             0 2003-03-03 03:00:00  12072.0  12162.0  12569.0   
4     Total             0 2003-03-03 04:00:00  12162.0  12569.0  13238.0   

   lags_45  lags_44  lags_43  lags_42  ...   lags_4   lags_3   lags_2  \
0  12072.0  12162.0  12569.0  13238.0  ...  15815.0  14745.0  13444.0   
1  12162.0  12569.0  13238.0  14191.0  ...  14745.0  13444.0  12350.0   
2  12569.0  13238.0  14191.0  15213.0  ...  13444.0  12350.0  11657.0   
3  13238.0  14191.0  15213.0  15646.0  ...  12350.0  11657.0  11357.0   
4  14191.0  15213.0  15646.0  15653.0  ...  11657.0  11357.0  11315.0   

    lags_1        y   post_1   post_2   post_3   post_4   post_5  
0  12350.0  11657.0  11357.0  11315.0

In [3]:
%%time
results = run_centralised(
    df=df2,
    lag_cols_reversed=lag_cols_reversed,
    forecast_horizon=forecast_horizon,
    input_size=args_INPUT,
    hidden_size=args_HIDDEN,
    num_layers=args_LAYERS,
    dropout=args_DROP,
    batch_size=args_BATCH_SIZE,
    epochs=args_EPOCHS,
    lr=args_LR,
    train_ratio=args_TRAIN_RATIO,
    partition_col='partition_id',
    device=None,
    clip_grad=1.0,
    verbose=True,
    lags=args_lags,
    fh=args_fh,
    time_col='ds',
    scaling_mode=args_SCALING_MODE,
    drop_boundary_gap=args_DROP_BOUNDARY_GAP,
    early_stop_enabled=args_EARLY_STOP_ENABLED,
    early_stop_tol=args_EARLY_STOP_TOL,
    min_epochs=args_MIN_EPOCHS,
    checkpoint_path=f'fh{args_fh+1}_cen_lstm.pt',
)

parts = results['parts']
train_idx = results['train_idx']
test_idx = results['test_idx']
dict_pred = results['dict_pred']
dict_true = results['dict_true']
dict_tr_pred = results['dict_tr_pred']
dict_tr_true = results['dict_tr_true']
dict_naive = results['dict_naive']

print('Device:', results['device'])
print('Stopped early:', results['stopped_early'])
print('Stop epoch:', results['stop_epoch'])
print('Final avg_normalised_loss:', results['final_avg_normalised_loss'])
print('Final avg_actual_loss:', results['final_avg_actual_loss'])
pd.DataFrame(results['round_logs']).tail()

[Centralised] epoch 001/350 | avg_normalised_loss=0.024716 | avg_actual_loss=22833.649807 | delta=nan | stopped=False
[Centralised] epoch 002/350 | avg_normalised_loss=0.019962 | avg_actual_loss=17949.306368 | delta=0.00475414 | stopped=False
[Centralised] epoch 003/350 | avg_normalised_loss=0.017729 | avg_actual_loss=15542.157066 | delta=0.00223298 | stopped=False
[Centralised] epoch 004/350 | avg_normalised_loss=0.017785 | avg_actual_loss=15709.709315 | delta=5.61966e-05 | stopped=False
[Centralised] epoch 005/350 | avg_normalised_loss=0.017346 | avg_actual_loss=15255.952753 | delta=0.000438932 | stopped=False
[Centralised] epoch 006/350 | avg_normalised_loss=0.015221 | avg_actual_loss=12945.315864 | delta=0.00212485 | stopped=False
[Centralised] epoch 007/350 | avg_normalised_loss=0.014855 | avg_actual_loss=12727.461898 | delta=0.0003664 | stopped=False
[Centralised] epoch 008/350 | avg_normalised_loss=0.014533 | avg_actual_loss=12214.708880 | delta=0.000322194 | stopped=False
[Cent

,epoch,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,stopped
301,302,0.007409,5408.093021,0.000030,False
302,303,0.007317,5554.655964,0.000093,False
303,304,0.007410,5527.539074,0.000093,False
304,305,0.007375,5499.712700,0.000035,False
305,306,0.007380,5499.691745,0.000005,True


In [4]:
# Base model and Naive baseline metrics before reconciliation
base_metrics = compute_metrics_from_dicts(dict_true, dict_pred, dict_train_true=dict_tr_true, h_idx=0)
naive_metrics = compute_metrics_from_dicts(dict_true, dict_naive, dict_train_true=dict_tr_true, h_idx=0)
display(base_metrics.tail())
display(naive_metrics.tail())

,partition_id,n_test,MAE,MASE,RMSSE
6,6,24830,9.424567,0.382215,0.411082
7,7,24830,20.909502,0.195969,0.204796
8,8,24830,16.283189,0.210909,0.216294
9,9,24830,15.997195,0.205373,0.209941
10,Overall,248300,25.174154,0.208980,0.215604


,partition_id,n_test,MAE,MASE,RMSSE
6,6,24830,23.280523,0.944147,0.890569
7,7,24830,99.788857,0.935247,0.924315
8,8,24830,70.846740,0.917648,0.888023
9,9,24830,70.184900,0.901039,0.876585
10,Overall,248300,130.509369,0.920088,0.898420


In [5]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

#path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/'
path = 'E:/yjz/Datasets/GEFCOM2017/'
S = pd.read_csv(f'{path}GEF_S.csv')
tags = pd.read_pickle(f'{path}tags.pkl')

def align_S_for_hierarchicalforecast(S, id_col="unique_id"):
    S = S.copy()
    if id_col not in S.columns:
        S = S.reset_index()
        if id_col not in S.columns:
            S = S.rename(columns={S.columns[0]: id_col})
    bottom_ids = [c for c in S.columns if c != id_col]
    agg_ids = [uid for uid in S[id_col].tolist() if uid not in bottom_ids]
    row_order = agg_ids + bottom_ids
    S_aligned = (
        S
        .set_index(id_col)
        .loc[row_order, bottom_ids]
        .reset_index())
    bottom_block = S_aligned[bottom_ids].tail(len(bottom_ids)).to_numpy()
    if not np.allclose(bottom_block, np.eye(len(bottom_ids))):
        raise ValueError("S Error")
    return S_aligned

S = align_S_for_hierarchicalforecast(S)

reconciliation_methods = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='wls_var'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]
reconcilers = HierarchicalReconciliation(reconcilers=reconciliation_methods)

recon_results = {}
train_frames = {}
test_frames = {}
for H_IDX in range(args_fh + 1):
    y_df, y_hat_df, train_frame, test_frame = make_reconciliation_frames(
        df=df2,
        train_idx=train_idx,
        test_idx=test_idx,
        parts=parts,
        dict_tr_pred=dict_tr_pred,
        dict_pred=dict_pred,
        forecast_horizon=forecast_horizon,
        base_model_name=args_MODEL_NAME,
        horizon_idx=H_IDX,
    )
    train_frames[H_IDX] = train_frame
    test_frames[H_IDX] = test_frame
    recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S_df=S, tags=tags, Y_df=y_df)

recon_results[0].head()

,unique_id,ds,y,clstm,clstm/BottomUp,clstm/MinTrace_method-mint_shrink,clstm/MinTrace_method-wls_var,clstm/MinTrace_method-ols,clstm/MinTrace_method-wls_struct
0,Total,2014-06-30 23:00:00,15138.0,15141.218572,15199.129306,15198.095782,15186.431398,15149.566036,15172.593559
1,Total,2014-07-01 00:00:00,13810.0,13836.395351,13827.416429,13825.708023,13827.840745,13834.467742,13829.958269
2,Total,2014-07-01 01:00:00,12979.0,13034.784830,13043.146697,13042.482378,13040.887491,13035.815558,13038.881694
3,Total,2014-07-01 02:00:00,12468.0,12509.704000,12495.406360,12498.576834,12500.950898,12508.631190,12504.411020
4,Total,2014-07-01 03:00:00,12226.0,12271.410962,12260.217311,12268.159312,12269.070761,12272.421592,12271.861510


In [6]:
# Save base, reconciled and naive forecasts; evaluate MAE, MASE and RMSSE
output_prefix = f'fh{args_fh+1}_cen'
eval_outputs = evaluate_reconciliation_results(
    recon_results=recon_results,
    train_frames=train_frames,
    test_frames=test_frames,
    forecast_horizon=forecast_horizon,
    base_model_name=args_MODEL_NAME,
    output_prefix=output_prefix,
    approach='centralised',
    round_logs=results['round_logs'],
    timings=results['timings'],
)

forecast_table = eval_outputs['forecast_table']
per_series_metrics = eval_outputs['per_series_metrics']
metrics_by_level = eval_outputs['metrics_by_level']
overall_metrics = eval_outputs['overall_metrics']
timing_df = eval_outputs['timing_df']
output_paths = eval_outputs['output_paths']

print('Saved outputs:')
for k, v in output_paths.items():
    print(f'  {k}: {v}')

metrics_by_level

Saved outputs:
  forecasts: fh6_cen_forecasts.csv
  per_series_metrics: fh6_cen_per_series_metrics.csv
  metrics_by_level: fh6_cen_metrics_by_level.csv
  overall_metrics: fh6_cen_overall_metrics.csv
  round_logs: fh6_cen_round_logs.csv
  timing: fh6_cen_timing.csv


,level,role,method,MAE,MASE,RMSSE,n_series,approach
0,1,top,base,217.653290,0.376230,0.394313,1,centralised
1,1,top,bu,217.100381,0.375275,0.387894,1,centralised
2,1,top,mint_ols,216.020605,0.373408,0.390269,1,centralised
3,1,top,mint_shrinkage,215.115263,0.371843,0.384909,1,centralised
4,1,top,mint_var,214.527675,0.370827,0.383793,1,centralised
5,1,top,naive,1632.296309,2.821548,2.639286,1,centralised
6,1,top,wls_structure,213.785456,0.369544,0.383664,1,centralised
7,2,middle,base,45.092667,0.503922,0.528336,6,centralised
8,2,middle,bu,44.865732,0.503049,0.527352,6,centralised
9,2,middle,mint_ols,45.892746,0.529546,0.552376,6,centralised


In [7]:
overall_metrics

,method,MAE,MASE,RMSSE,n_series,approach
0,base,62.182499,0.494559,0.520311,10,centralised
1,bu,61.991047,0.493940,0.519078,10,centralised
2,mint_ols,62.530442,0.510010,0.534251,10,centralised
3,mint_shrinkage,61.612849,0.491775,0.516309,10,centralised
4,mint_var,61.511282,0.491729,0.516236,10,centralised
5,naive,402.402792,2.813179,2.624031,10,centralised
6,wls_structure,61.563120,0.493383,0.516659,10,centralised


In [8]:
timing_df

,module,seconds
0,training_sec,18346.560206
1,prediction_sec,21.262793
2,total_sec,18380.655532
